In [18]:
from PIL import Image
import torch
import torch.nn.functional as F
import pickle
import os
import timm
import pandas as pd
import numpy as np
import torch.nn as nn
from collections import defaultdict
from torchvision import transforms
from google.colab import drive
drive.mount('/content/drive')
device = "cuda" if torch.cuda.is_available() else "cpu"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
# Copying 8k to Colabs local SSD
if not os.path.exists("/content/flickr8k.zip"):
  !cp -r "/content/drive/MyDrive/MMRetrieval/flickr8k.zip" /content/

In [20]:
if not os.path.isdir("/content/flickr8k/Images"):
  !unzip "/content/flickr8k.zip" -d "/content/flickr8k"

In [21]:
# Copying 30k to Colabs local SSD
if not os.path.exists("/content/flickr30k.zip"):
  !cp -r "/content/drive/MyDrive/MMRetrieval/flickr30k.zip" /content/

In [22]:
if not os.path.isdir("/content/flickr30k/Images"):
  !unzip "/content/flickr30k.zip" -d "/content/flickr30k"

In [23]:
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [24]:
from torch.utils.data import Dataset, DataLoader

class ImageEmbeddingDataset(Dataset):

    def __init__(self, image_caption_map, image_transform, image_dir):

        self.image_transform = image_transform
        self.image_dir = image_dir

        # Directly get unique image paths (filenames) from the keys of image_caption_map
        self.image_paths = list(image_caption_map.keys())

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):

        filename = self.image_paths[idx]
        full_path = os.path.join(self.image_dir, filename)

        image = Image.open(full_path).convert("RGB")

        image = self.image_transform(image)

        return image, full_path

In [25]:
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F


class ViTEncoder(nn.Module):

    def __init__(
        self,
        embed_dim=512,
        freeze_backbone=True,
        dropout=0.15
    ):
        super().__init__()

        self.freeze_backbone = freeze_backbone

        # --------------------------------------------------
        # ViT-B/16
        # Original ImageNet pretrained ViT-B/16
        # --------------------------------------------------

        self.vit = timm.create_model(
            "vit_base_patch16_224",
            pretrained=True,
            num_classes=0
        )

        # --------------------------------------------------
        # Freeze / unfreeze backbone
        # --------------------------------------------------

        for param in self.vit.parameters():
            param.requires_grad = not freeze_backbone

        # --------------------------------------------------
        # Projection Head
        # 768 -> 1024 -> 512
        # --------------------------------------------------

        self.projection = nn.Sequential(

            nn.Linear(768, 1024),

            nn.LayerNorm(1024),

            nn.GELU(),

            nn.Dropout(dropout),

            nn.Linear(1024, embed_dim),

            nn.LayerNorm(embed_dim)
        )

        # --------------------------------------------------
        # Initialization
        # --------------------------------------------------

        for module in self.projection:

            if isinstance(module, nn.Linear):

                nn.init.xavier_uniform_(
                    module.weight
                )

                nn.init.zeros_(
                    module.bias
                )

    def forward(self, images):

        # --------------------------------------------------
        # ViT feature extraction
        # --------------------------------------------------

        if self.freeze_backbone:

            with torch.no_grad():

                features = self.vit.forward_features(
                    images
                )

        else:

            features = self.vit.forward_features(
                images
            )

        # --------------------------------------------------
        # ViT output
        #
        # (B, 197, 768)
        #
        # 1 CLS token
        # 196 patch tokens
        # --------------------------------------------------

        cls_token = features[:, 0]

        # --------------------------------------------------
        # Patch tokens
        # --------------------------------------------------

        patch_tokens = features[:, 1:]

        # --------------------------------------------------
        # Mean pooled patch representation
        # --------------------------------------------------

        patch_mean = patch_tokens.mean(
            dim=1
        )

        # --------------------------------------------------
        # Combine CLS + patch information
        # --------------------------------------------------

        features = (
            cls_token +
            patch_mean
        ) / 2.0

        # --------------------------------------------------
        # Projection
        # --------------------------------------------------

        embeddings = self.projection(
            features
        )

        # --------------------------------------------------
        # L2 normalization
        # --------------------------------------------------

        embeddings = F.normalize(
            embeddings,
            p=2,
            dim=1
        )

        return embeddings

In [26]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

from torch.nn.utils.rnn import (
    pack_padded_sequence,
    pad_packed_sequence
)


class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, embed_dim)
        position = torch.arange(0, max_len).unsqueeze(1).float()

        div_term = torch.exp(
            torch.arange(0, embed_dim, 2).float()
            * (-math.log(10000.0) / embed_dim)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)      # (1, max_len, embed_dim)

        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class TextEncoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        embed_dim=512,
        num_heads=8,
        num_layers=4,
        ff_dim=2048,
        pad_idx=0,
        dropout=0.15
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=pad_idx
        )

        self.position = PositionalEncoding(
            embed_dim
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.projection = nn.Sequential(
            nn.Linear(embed_dim, 1024),
            nn.LayerNorm(1024),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(1024, 512),
            nn.LayerNorm(512)
        )

        for module in self.projection:

            if isinstance(module, nn.Linear):

                nn.init.xavier_uniform_(
                    module.weight
                )

                nn.init.zeros_(
                    module.bias
                )

    def forward(
        self,
        captions,
        lengths
    ):

        x = self.embedding(captions)

        x = self.position(x)

        device = captions.device

        max_len = captions.size(1)

        lengths = lengths.to(device)

        mask = (
            torch.arange(
                max_len,
                device=device
            )
            .unsqueeze(0)
            .expand(captions.size(0), -1)
            >= lengths.unsqueeze(1)
        )

        x = self.transformer(
            x,
            src_key_padding_mask=mask
        )

        # Masked mean pooling

        valid_mask = (~mask).unsqueeze(-1).float()

        x = (
            x * valid_mask
        ).sum(dim=1) / valid_mask.sum(
            dim=1
        ).clamp(min=1e-8)

        embeddings = self.projection(x)

        embeddings = F.normalize(
            embeddings,
            p=2,
            dim=1
        )

        return embeddings

In [27]:
#Joint model
class ViTTransformerRetrieval(nn.Module):

    def __init__(self, image_encoder, text_encoder):
        super().__init__()

        self.image_encoder = image_encoder
        self.text_encoder = text_encoder

        # Learnable CLIP-style temperature
        self.logit_scale = nn.Parameter(
            torch.tensor(np.log(1 / 0.07), dtype=torch.float32)
        )

    def forward(self, images, captions, lengths):

        image_emb = self.image_encoder(images)

        text_emb = self.text_encoder(
            captions,
            lengths
        )

        return image_emb, text_emb

In [28]:
def generate_image_embeddings(loader,model):

    model.eval()


    image_embeddings = []
    image_paths = []

    with torch.no_grad():

        for images, paths in loader:

            images = images.to(device, non_blocking=True)

            embeddings = model.image_encoder(images)

            image_embeddings.append(
                embeddings.cpu()
            )

            image_paths.extend(paths)

    image_embeddings = torch.cat(
        image_embeddings,
        dim=0
    )

    return image_embeddings, image_paths

In [29]:
def encode_caption(text, vocab):

    tokens = text.lower().strip().split()

    encoded = [vocab["<SOS>"]]

    for token in tokens:
        encoded.append(
            vocab.get(token, vocab["<UNK>"])
        )

    encoded.append(vocab["<EOS>"])

    return encoded

In [30]:
def generate_caption_embeddings(model, split_image_filenames, main_caption_map, vocab):

    model.eval()

    caption_embeddings = []

    captions = []

    caption_to_image = []

    image_index = 0

    with torch.no_grad():

        for split_name in ["train", "val", "test"]:

            for image_filename in split_image_filenames[split_name]:

                captions_for_this_image = main_caption_map[image_filename]

                for caption_text in captions_for_this_image:

                    encoded = encode_caption(
                        caption_text,
                        vocab
                    )

                    caption_tensor = torch.tensor(
                        encoded,
                        dtype=torch.long
                    ).unsqueeze(0).to(device)

                    # Convert lengths to a tensor
                    lengths = torch.tensor([len(encoded)], dtype=torch.long,device=device)

                    embedding = model.text_encoder(
                        caption_tensor,
                        lengths
                    )

                    caption_embeddings.append(
                        embedding.cpu()
                    )

                    captions.append(caption_text)

                    caption_to_image.append(image_index)

                image_index += 1

    caption_embeddings = torch.cat(
        caption_embeddings,
        dim=0
    )

    return (
        caption_embeddings,
        captions,
        caption_to_image
    )

In [31]:
def build_r5(vocab):
  image_encoder = ViTEncoder(freeze_backbone=True).to(device)

  text_encoder = TextEncoder(
      vocab_size=len(vocab),
      embed_dim=512,
      pad_idx=vocab["<PAD>"]
  ).to(device)

  model = ViTTransformerRetrieval(
      image_encoder=image_encoder,
      text_encoder=text_encoder
  ).to(device)

  return model


In [32]:
def image_caption_map(main_df):

    main_caption_map = defaultdict(list)

    bad_img = "861608773_bdafd5c996.jpg"

    for _, row in main_df.iterrows():

        if row["image"] == bad_img:
            continue

        main_caption_map[row["image"]].append(row["caption"])

    return main_caption_map

In [33]:
def generate_database(MODEL,DATASET):
  if DATASET == "flickr8k":
    split_file = "/content/drive/MyDrive/MMRetrieval/Preprocessing/flickr8k/flickr8k_split.pkl"
    vocab_file = "/content/drive/MyDrive/MMRetrieval/Preprocessing/flickr8k/vocab.pkl"
    CAPTION_FILE = "/content/drive/MyDrive/MMRetrieval/flickr8k/captions.txt"
    IMAGE_DIR = "/content/flickr8k/Images"
  else:
    split_file = "/content/drive/MyDrive/MMRetrieval/Preprocessing/flickr30k/flickr30k_split.pkl"
    vocab_file = "/content/drive/MyDrive/MMRetrieval/Preprocessing/flickr30k/vocab.pkl"
    CAPTION_FILE = "/content/drive/MyDrive/MMRetrieval/flickr30k/captions.txt"
    IMAGE_DIR = "/content/flickr30k/Images"

  with open(split_file, "rb") as f:
        split = pickle.load(f)
  with open(vocab_file, "rb") as f:
        vocab = pickle.load(f)

  df = pd.read_csv(CAPTION_FILE)

  train_imgs = split["train"]
  val_imgs = split["val"]
  test_imgs = split["test"]

  train_df = df[df["image"].isin(train_imgs)].reset_index(drop=True)
  val_df = df[df["image"].isin(val_imgs)].reset_index(drop=True)
  test_df = df[df["image"].isin(test_imgs)].reset_index(drop=True)

  train_df = train_df.dropna(subset=["caption"]).reset_index(drop=True)
  val_df = val_df.dropna(subset=["caption"]).reset_index(drop=True)
  test_df = test_df.dropna(subset=["caption"]).reset_index(drop=True)

  main_df = pd.concat([train_df, val_df, test_df],ignore_index=True)

  main_caption_map = image_caption_map(main_df)
  print(len(main_caption_map))
  print(next(iter(main_caption_map.items())))

  dataset = ImageEmbeddingDataset(
    main_caption_map,
    image_transform,
    IMAGE_DIR
  )

  loader = DataLoader(
      dataset,
      batch_size=256,      # 64 or 128 depending on GPU memory
      shuffle=False,
      num_workers=8,
      pin_memory=True
  )



  ckpt=f"/content/drive/MyDrive/MMRetrieval/{MODEL}/{DATASET}/best_retrieval_{MODEL}_model.pth"

  checkpoint=torch.load(
      ckpt,
      map_location=device
  )
  if MODEL == "R5":
    model = build_r5(vocab)


  model.load_state_dict(checkpoint["model_state_dict"])

  image_embeddings, image_paths = generate_image_embeddings(loader,model)
  caption_embeddings, captions, caption_to_image = generate_caption_embeddings(model, split, main_caption_map, vocab)

  # --------------- ----------
  # Save
  # -------------------------
  save_dir = f"/content/drive/MyDrive/MMRetrieval/{MODEL}/{DATASET}/retrieval_db"

  os.makedirs(save_dir, exist_ok=True)

  torch.save(
      image_embeddings,
      os.path.join(save_dir, "image_embeddings.pt")
  )

  torch.save(
      caption_embeddings,
      os.path.join(save_dir, "caption_embeddings.pt")
  )

  with open(os.path.join(save_dir, "image_paths.pkl"), "wb") as f:
      pickle.dump(image_paths, f)

  with open(os.path.join(save_dir, "captions.pkl"), "wb") as f:
      pickle.dump(captions, f)

  with open(os.path.join(save_dir, "caption_to_image.pkl"), "wb") as f:
      pickle.dump(caption_to_image, f)

  print("Database created successfully.")
  print("Images   :", len(image_paths))
  print("Captions :", len(captions))

In [34]:
#generate_database("R4","flickr8k")
#generate_database("R4","flickr30k")
generate_database("R5","flickr8k")
generate_database("R5","flickr30k")
#generate_database("R3","flickr8k")
#generate_database("R3","flickr30k")
#generate_database("R2","flickr8k")
#generate_database("R2","flickr30k")


8090
('1001773457_577c3a7d70.jpg', ['A black dog and a spotted dog are fighting', 'A black dog and a tri-colored dog playing with each other on the road .', 'A black dog and a white dog with brown spots are staring at each other in the street .', 'Two dogs of different breeds looking at each other on the road .', 'Two dogs on pavement moving toward each other .'])


/tmp/ipykernel_2938/664705790.py:69: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Database created successfully.
Images   : 8090
Captions : 40450
31783
('1000092795.jpg', [' Two young guys with shaggy hair look at their hands while hanging out in the yard .', ' Two young , White males are outside near many bushes .', ' Two men in green shirts are standing in a yard .', ' A man in a blue shirt standing in a garden .', ' Two friends enjoy time spent together .'])


/tmp/ipykernel_2938/664705790.py:69: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Database created successfully.
Images   : 31783
Captions : 158914
